In [ ]:
# ============================================================
# Notebook 3 Cell 1: Setup for ablation experiment on L4
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/cs4782_matcher/Matcher

!pip install -q matplotlib torchmetrics torchshow opencv-python timm POT omegaconf iopath tqdm future tensorboardX
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import os, shutil
FSS_ROOT = '/content/datasets/FSS-1000'
if not os.path.exists(os.path.join(FSS_ROOT, 'data')) or \
   len(os.listdir(os.path.join(FSS_ROOT, 'data'))) < 1000:
    print("Extracting FSS-1000...")
    os.makedirs('/content/datasets', exist_ok=True)
    !unzip -q -o /content/drive/MyDrive/cs4782_matcher/Matcher/datasets/FSS-1000.zip -d /content/datasets/

    data_dir = os.path.join(FSS_ROOT, 'data')
    os.makedirs(data_dir, exist_ok=True)
    cats = [d for d in os.listdir(FSS_ROOT)
            if os.path.isdir(os.path.join(FSS_ROOT, d)) and d not in ('data', 'splits')]
    for cat in cats:
        src = os.path.join(FSS_ROOT, cat)
        dst = os.path.join(data_dir, cat)
        if not os.path.exists(dst):
            shutil.move(src, dst)

    splits_dir = os.path.join(FSS_ROOT, 'splits')
    os.makedirs(splits_dir, exist_ok=True)
    for f in ['test.txt', 'trn.txt', 'val.txt']:
        dst = os.path.join(splits_dir, f)
        if not os.path.exists(dst):
            !wget -q https://raw.githubusercontent.com/juhongm999/hsnet/main/data/splits/fss/{f} -O {dst}

assert len(os.listdir('/content/datasets/FSS-1000/data')) == 1000
print("=== Setup complete ===")
!nvidia-smi | grep -E "L4|T4|MiB" | head -2

In [ ]:
%cd /content/drive/MyDrive/cs4782_matcher/Matcher

# Back up (only if not already backed up)
!test -f matcher/Matcher.py.orig || cp matcher/Matcher.py matcher/Matcher.py.orig

# Insert a conditional skip right after the retain_ind line.
# If env var DISABLE_REVERSE is set, make retain_ind all True (skip reverse filtering).
# This is safer than removing the line, because the main experiment will still run identically
# unless you set DISABLE_REVERSE=1 on the command line.

import re
with open('matcher/Matcher.py.orig', 'r') as f:
    src = f.read()

old = "        retain_ind = torch.isin(indices_reverse[1], indices_mask)\n"
new = (
    "        retain_ind = torch.isin(indices_reverse[1], indices_mask)\n"
    "        # Ablation hook: when env var DISABLE_REVERSE=1, bypass reverse-matching filtering\n"
    "        # to reproduce the 'forward-only' ablation from Matcher paper Table 4b.\n"
    "        import os as _os\n"
    "        if _os.environ.get('DISABLE_REVERSE', '0') == '1':\n"
    "            retain_ind = torch.ones_like(retain_ind, dtype=torch.bool)\n"
)

assert old in src, "Could not find the target line to patch!"
patched = src.replace(old, new, 1)

with open('matcher/Matcher.py', 'w') as f:
    f.write(patched)

# Verify the patch
!grep -n "DISABLE_REVERSE\|retain_ind = torch" matcher/Matcher.py | head

In [ ]:
# ============================================================
# Notebook 3 Cell 2 (FORWARD-ONLY ABLATION):
# Run ablation with bidirectional matching disabled
# Reproduces Table 4b "forward" row (81.1 mIoU in paper)
#
# KEY CHANGE vs main experiment:
#   - Env var DISABLE_REVERSE=1 skips reverse-matching filtering
#   - All other params identical to main (alpha=0.8, beta=0.2)
# ============================================================
%cd /content/drive/MyDrive/cs4782_matcher/Matcher

!DISABLE_REVERSE=1 stdbuf -oL -eL python -u main_oss.py \
    --benchmark fss \
    --datapath /content/datasets \
    --max_sample_iterations 30 \
    --sample-range "(4,6)" \
    --multimask_output 0 \
    --alpha 0.8 --beta 0.2 --exp 1. \
    --num_merging_mask 10 \
    --fold 0 \
    --log-root "output/fss/fold0_forward_only" 2>&1 | tee /content/drive/MyDrive/cs4782_matcher/results/run_ablation_live.log

In [ ]:
# ============================================================
# Notebook 3 Cell 3 (FORWARD-ONLY): Back up ablation results
# ============================================================
import os, shutil, glob, json, re

PROJECT_ROOT = '/content/drive/MyDrive/cs4782_matcher'
MATCHER_ROOT = os.path.join(PROJECT_ROOT, 'Matcher')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')

os.makedirs(os.path.join(RESULTS_DIR, 'logs'), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'raw_ablation_run'), exist_ok=True)

pattern = os.path.join(MATCHER_ROOT, 'output/fss/fold0_forward_only/_TEST_*.log/log.txt')
candidates = glob.glob(pattern)
if not candidates:
    pattern2 = os.path.join(MATCHER_ROOT, 'output/fss/fold0_forward_only/**/log.txt')
    candidates = glob.glob(pattern2, recursive=True)

assert len(candidates) > 0, "ERROR: Could not find ablation log.txt!"
abl_log_src = max(candidates, key=os.path.getmtime)
print(f"Found ablation log: {abl_log_src}")
print(f"Log size: {os.path.getsize(abl_log_src)} bytes")

# Save as ablation_forward_only.log (clearer name)
abl_log_dst = os.path.join(RESULTS_DIR, 'logs', 'ablation_forward_only.log')
shutil.copy(abl_log_src, abl_log_dst)

live_log = '/content/drive/MyDrive/cs4782_matcher/results/run_ablation_live.log'
if os.path.exists(live_log):
    shutil.copy(live_log, os.path.join(RESULTS_DIR, 'logs', 'ablation_forward_only_live.log'))

log_dir_src = os.path.dirname(abl_log_src)
log_dir_dst = os.path.join(RESULTS_DIR, 'raw_ablation_run')
if os.path.exists(log_dir_dst):
    shutil.rmtree(log_dir_dst)
shutil.copytree(log_dir_src, log_dir_dst)

with open(abl_log_dst, 'r') as f:
    text = f.read()
fold_match = re.search(r'Fold\s+\d+\s+mIoU:\s+([\d.]+)\s+FB-IoU:\s+([\d.]+)', text)
assert fold_match, "ERROR: Could not parse final mIoU!"
final_miou   = float(fold_match.group(1))
final_fb_iou = float(fold_match.group(2))

summary = {
    'experiment': 'ablation: forward-only matching (bidirectional disabled)',
    'description': 'Table 4b "forward" row: only forward matching, no reverse filtering',
    'benchmark': 'FSS-1000 fold 0',
    'episodes': 2400,
    'final_mIoU': final_miou,
    'final_FB_IoU': final_fb_iou,
    'paper_reported_mIoU': 81.1,
    'gpu': 'L4',
}
with open(os.path.join(RESULTS_DIR, 'ablation_experiment_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("ABLATION (FORWARD-ONLY) RESULTS SAFELY BACKED UP")
print("="*60)
print(f"  Final mIoU:   {final_miou:.2f}  (paper: 81.1)")
print(f"  Final FB-IoU: {final_fb_iou:.2f}")

!ls -la /content/drive/MyDrive/cs4782_matcher/results/logs/